# 내 RTX 3050 GPU 시작 점검

위 오른쪽 커널이 **LLM GPU (RTX 3050)** 인지 확인하세요. 이 노트북은 설치된 패키지만 사용하며 모델이나 API 키를 다운로드하지 않습니다.

## 1. GPU와 패키지 확인

Hugging Face·PyTorch 실습은 이 커널을 사용합니다. OpenAI·Upstage·Pinecone API 실습은 기존 **llm(ipykernel)** 커널을 사용하세요.

In [ ]:
import sys
import torch
import transformers
import tokenizers
import accelerate

print('Python:', sys.executable)
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('Transformers:', transformers.__version__, 'Tokenizers:', tokenizers.__version__)
assert torch.cuda.is_available(), 'GPU가 보이지 않습니다. LLM GPU 커널을 선택하세요.'
device = torch.device('cuda:0')
print('GPU:', torch.cuda.get_device_name(0))

## 2. GPU에서 실제 계산

RTX 3050의 메모리는 4GB입니다. 작은 행렬로 계산이 GPU에서 이루어지는지 확인합니다. `high` 설정은 일부 float32 행렬 계산을 빠르게 하는 대신 정밀도를 조금 바꿀 수 있습니다. 정확한 수치 비교가 필요하면 해당 설정을 사용하지 마세요.

In [ ]:
torch.set_float32_matmul_precision('high')
free_bytes, total_bytes = torch.cuda.mem_get_info()
print(f'GPU 메모리: 사용 가능 {free_bytes / 2**30:.2f}GB / 전체 {total_bytes / 2**30:.2f}GB')
with torch.inference_mode():
    a = torch.randn((1024, 1024), device=device, dtype=torch.float16)
    result = a @ a
    torch.cuda.synchronize()
    print('계산 결과 장치:', result.device)
del a, result
torch.cuda.empty_cache()

## 수업별 커널 선택

- `ch1_허깅페이스.ipynb`: **LLM GPU (RTX 3050)**
- 9장 OpenAI·Upstage·Pinecone RAG: **llm(ipykernel)**. 이 API들은 서버에서 계산하므로 내 GPU 사용률이 오르지 않는 것이 정상입니다.
- TensorFlow 딥러닝: **ml-dl-nlp**. 이 환경에서도 GPU 계산을 확인했습니다.
- Ollama: 모델이 자체적으로 GPU를 사용합니다. `llama3.2:1b`는 100% GPU 실행을 확인했습니다.

4GB 메모리에서는 큰 모델을 동시에 여러 개 띄우지 마세요. 메모리가 부족하면 Ollama 모델을 내리거나 Jupyter 커널을 재시작하세요.